# synthetic-gov-data-kit — End-to-End CLI Workflow

This notebook walks through the full `govsynth` CLI from installation through generation,
validation, and inspection. If you prefer the command line over the Python API — for
scripting, CI/CD pipelines, or quick exploration — the CLI is the right entry point.

**What you'll do:**

1. Check installation and list available presets
2. Generate test cases to a directory
3. Generate with a profile strategy
4. Stream cases to stdout (pipe-friendly)
5. Validate generated output files
6. Inspect a single case
7. Run a multi-preset batch
8. Verify bundled threshold data
9. Use `--json` output for scripting with `jq`
10. Put it all together in a one-liner pipeline

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

## 1. Check Installation and List Presets

Confirm the CLI is on your `PATH` and see which presets ship out of the box.

In [ ]:
!govsynth --help

In [ ]:
!govsynth list-presets

## 2. Generate Cases to a Directory

`govsynth generate` writes one YAML file per case when given `--output`.
Passing `--seed` makes the run reproducible.

In [ ]:
import os
os.makedirs('./cli_output', exist_ok=True)

In [ ]:
!govsynth generate snap.va --n 10 --seed 42 --output ./cli_output/snap_va/

In [ ]:
# Confirm files were written
import glob
files = sorted(glob.glob('./cli_output/snap_va/*.yaml'))
print(f'{len(files)} YAML files written:')
for f in files:
    print(' ', f)

## 3. Generate with a Profile Strategy

The `--profile-strategy` flag controls how synthetic household profiles are sampled.
Available strategies: `edge_saturated` (default), `realistic`, `uniform`, `adversarial`.

`realistic` draws income and household-size distributions from ACS census data, producing
cases that reflect the actual applicant population rather than saturating policy boundaries.

In [ ]:
!govsynth generate snap.va --n 5 --seed 42 --profile-strategy realistic --output ./cli_output/realistic/

## 4. Stream to Stdout (Pipe-Friendly)

Omit `--output` and the cases stream to stdout. Combine with `head`, `jq`, or any
Unix pipeline. Supported stdout formats are `yaml` (default) and `jsonl`.

In [ ]:
# Stream 3 cases as JSONL, show only the first 2 lines
!govsynth generate snap.va --n 3 --seed 42 --format jsonl | head -2

In [ ]:
# Stream as YAML (default format) — use head to cap output
!govsynth generate snap.va --n 2 --seed 42 | head -30

## 5. Validate Generated Files

`govsynth validate` checks output files against the `TestCase` schema and reports
pass/fail counts. It auto-detects YAML, JSONL, and CSV by extension.

Pass `--json` to get machine-readable output suitable for CI assertions.

In [ ]:
# Validate the first generated file
import glob
snap_files = sorted(glob.glob('./cli_output/snap_va/*.yaml'))
if snap_files:
    !govsynth validate {snap_files[0]}

In [ ]:
# Validate with JSON output — useful in CI scripts
if snap_files:
    !govsynth validate {snap_files[0]} --json

## 6. Inspect a Specific Case

`govsynth show` pretty-prints a single case from a YAML file. Pass a `case_id` as the
second argument to select a specific case; omit it to show the first one.

Use `--raw` to get plain YAML on stdout, or `--json` for JSON.

In [ ]:
import glob
files = sorted(glob.glob('./cli_output/snap_va/*.yaml'))
print('Available case files:')
for f in files:
    print(' ', f)

In [ ]:
# Show the first case in the first file (rich-formatted in the terminal, raw YAML here)
if files:
    !govsynth show {files[0]} --raw

In [ ]:
# Show as JSON — useful when piping to jq or other tools
if files:
    !govsynth show {files[0]} --json

## 7. Run a Multi-Preset Batch

`govsynth batch` generates cases across multiple presets in a single command.
Pass each preset with `--preset` (the flag is repeatable). All cases land in the
same output directory; the seed is offset per preset for diversity.

In [ ]:
!govsynth batch --preset snap.va --preset snap.ca --n 5 --seed 42 --output ./cli_output/batch/

In [ ]:
# Count output files by jurisdiction
import glob
batch_files = glob.glob('./cli_output/batch/*.yaml')
print(f'{len(batch_files)} total YAML files in batch output')

## 8. Verify Threshold Data

`govsynth verify-thresholds` checks the `_metadata.verification_status` field in every
bundled threshold JSON file. Use `--program` to filter to a single program.

Valid program keys: `snap`, `wic`, `medicaid`, `us_fpl`.

In [ ]:
# Check all bundled threshold files
!govsynth verify-thresholds

In [ ]:
# Filter to SNAP only
!govsynth verify-thresholds --program snap

In [ ]:
# Machine-readable JSON output
!govsynth verify-thresholds --program snap --json

## 9. JSON Output for Scripting

Every command that emits status supports `--json`. This produces a stable, machine-readable
envelope useful for CI assertions, log ingestion, or piping to `jq`.

In [ ]:
# Generate and capture structured status
!govsynth generate snap.va --n 5 --seed 42 --output ./cli_output/json_demo/ --json

In [ ]:
# Pipe generate status to jq to extract the case count
# (requires jq installed; falls back gracefully if not)
!govsynth generate snap.va --n 5 --seed 42 --output ./cli_output/json_demo/ --json --quiet | jq '.n' 2>/dev/null || echo '(jq not installed — raw output above)'

In [ ]:
# Pipe validate status to jq — check .status field in a script
import glob
demo_files = sorted(glob.glob('./cli_output/json_demo/*.yaml'))
if demo_files:
    !govsynth validate {demo_files[0]} --json | jq '.status' 2>/dev/null || echo '(jq not installed)'

## 10. Putting It All Together

A full generate-validate-inspect pipeline in three shell commands — ready to drop into
a `Makefile`, CI job, or shell script.

In [ ]:
# One-liner: generate → validate → show first case
# Each step exits non-zero on failure, so chaining with && is safe in scripts.

!govsynth generate snap.va --n 10 --seed 42 --output ./cli_output/pipeline/ --quiet && \
 govsynth validate $(ls ./cli_output/pipeline/*.yaml | head -1) --quiet && \
 govsynth show $(ls ./cli_output/pipeline/*.yaml | head -1) --raw | head -20

### CI snippet

```bash
#!/usr/bin/env bash
set -euo pipefail

OUT=./ci_output

# Generate
govsynth generate snap.va --n 50 --seed 42 --output "$OUT/snap_va/" --quiet

# Validate every file — exits 1 if any case fails schema check
for f in "$OUT"/snap_va/*.yaml; do
  govsynth validate "$f" --quiet
done

# Verify threshold data freshness
govsynth verify-thresholds --json | jq -e '.status == "ok"'

echo "All checks passed."
```

---

**Next steps:**
- See `notebooks/01_quickstart.ipynb` for the Python API equivalent of this workflow
- See `notebooks/04_batch_pipeline.ipynb` for multi-program batch generation
- Run `govsynth --help` or `govsynth <command> --help` for full flag reference